# 02 - Classificação: Regressão Logística, LDA, QDA e KNN

## 1. Por que não usar Regressão Linear comum (OLS) para Classificação?

Em problemas de **Classificação**, a variável resposta ($Y$) é categórica (ex.: $Y=1$ para caloteiro/inadimplente e $Y=0$ para adimplente/bom pagador).

Se tentarmos ajustar uma reta de regressão linear comum ($Y = \beta_0 + \beta_1 X$):
1. **Previsões sem sentido:** Para valores muito baixos de $X$, a reta cospe números negativos (ex.: $-0{,}20$). Para valores altos, cospe valores acima de $100\%$ (ex.: $1{,}35$). Na prática, uma probabilidade **só pode existir entre $0$ e $1$ ($0\%$ e $100\%$)**.
2. **Variância dos erros não constante:** Como os pontos reais são apenas $0$ ou $1$, os resíduos nunca formam uma distribuição normal homogênea (violação de homocedasticidade).

---

## 2. A Solução: A Função Sigmoide e a Regressão Logística

Em vez de tentar prever uma linha reta infinita, a Regressão Logística passa a conta da reta por uma função em formato de "S" chamada **Função Sigmoide (ou Logística)**. Ela funciona como um **esmagador universal**: pega qualquer número real de $-\infty$ a $+\infty$ e comprime obrigatoriamente para a faixa de $(0, 1)$:

$$p(X) = P(Y=1 \mid X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}} = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

---

### O que é a letra "e" e de onde ela vem?

A letra **$e$** na fórmula não é uma incógnita ou variável; ela é uma **constante matemática fixa**, exatamente como o número Pi ($\pi \approx 3{,}1415$). 

* O nome dela é **Número de Euler** (ou base dos logaritmos naturais).
* O seu valor numérico aproximado é:
  $$e \approx 2{,}71828$$
* **Como calcular $e^z$ na prática?**
  Significa pegar esse número $2{,}71828$ e elevar ao expoente $z$ (o resultado da reta):
  * Se $z = 0$: $e^0 = 1$ (qualquer base elevada a zero resulta em $1$).
  * Se $z = 1$: $e^1 \approx 2{,}718$.
  * Se $z = 0{,}5$: $e^{0{,}5} = \sqrt{2{,}71828} \approx 1{,}6487$.
  * Se $z$ for negativo (ex.: $z = -5$): $e^{-5} = \frac{1}{e^5} = \frac{1}{2{,}71828^5} = \frac{1}{148{,}41} \approx 0{,}0067$.

> Em linguagens como Python ou em calculadoras, basta usar a função `np.exp(z)` ou a tecla `exp`.

---

### Para que serve a Sigmoide? (A Intuição da Curva em S)

A sigmoide garante que o resultado seja sempre uma probabilidade válida:
* Se a conta da reta der um valor muito negativo ($z \to -\infty$): $e^z \to 0$, logo $p(X) \to \mathbf{0}$ ($0\%$ de probabilidade).
* Se a conta da reta der exatamente zero ($z = 0$): $p(X) = \frac{1}{1 + e^0} = \frac{1}{1 + 1} = \frac{1}{2} = \mathbf{0{,}5}$ ($50\%$ de probabilidade / ponto de corte neutro).
* Se a conta da reta der um valor muito positivo ($z \to +\infty$): $p(X) \to \mathbf{1}$ ($100\%$ de probabilidade).

---

### Exemplo Prático com Números: Calculando na mão passo a passo

Suponha que um modelo treinado aprendeu os seguintes parâmetros para prever inadimplência pelo saldo (`balance`):
* $\beta_0 = -10{,}5$ (intercepto fixo)
* $\beta_1 = 0{,}0055$ (peso por dólar de dívida)

#### Caso 1: Cliente A (Saldo baixo: $X = \$1.000$)
1. **Passo 1 (Cálculo da reta linear $z$):**
   $$z = \beta_0 + \beta_1 X = -10{,}5 + (0{,}0055 \times 1000) = -10{,}5 + 5{,}5 = -5{,}0$$
2. **Passo 2 (Cálculo da exponencial $e^z$ com o número de Euler):**
   $$e^{-5{,}0} = 2{,}71828^{-5} \approx 0{,}006737$$
3. **Passo 3 (Aplicação na fórmula da Sigmoide):**
   $$p(X) = \frac{e^z}{1 + e^z} = \frac{0{,}006737}{1 + 0{,}006737} = \frac{0{,}006737}{1{,}006737} \approx \mathbf{0{,}0067} \quad (\mathbf{0{,}67\%} \text{ de risco})$$
   *Conclusão:* Como o saldo é baixo, o risco de inadimplência é desprezível (menor que $1\%$).

#### Caso 2: Cliente B (Saldo alto: $X = \$2.000$)
1. **Passo 1 (Cálculo da reta linear $z$):**
   $$z = \beta_0 + \beta_1 X = -10{,}5 + (0{,}0055 \times 2000) = -10{,}5 + 11{,}0 = +0{,}5$$
2. **Passo 2 (Cálculo da exponencial $e^z$ com o número de Euler):**
   $$e^{+0{,}5} = 2{,}71828^{0{,}5} \approx 1{,}6487$$
3. **Passo 3 (Aplicação na fórmula da Sigmoide):**
   $$p(X) = \frac{e^z}{1 + e^z} = \frac{1{,}6487}{1 + 1{,}6487} = \frac{1{,}6487}{2{,}6487} \approx \mathbf{0{,}6225} \quad (\mathbf{62{,}25\%} \text{ de risco})$$
   *Conclusão:* Com a dívida acumulada em $\$2.000$, a probabilidade saltou para mais de $62\%$.

---

## 3. O que são Chances (*Odds*) e Razão de Chances (*Odds Ratio*)?

Em estatística e apostas, **Odds (Chances)** é a razão entre a probabilidade de o evento acontecer e a de não acontecer:

$$\text{Odds} = \frac{p(X)}{1 - p(X)}$$

Isolando a combinação linear na fórmula da sigmoide, chegamos ao **Logit** (o logaritmo natural das chances):

$$\log\left(\frac{p(X)}{1 - p(X)}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$$

* **Interpretação do coeficiente $\beta_j$:** Um aumento de 1 unidade em $X_j$ multiplica as chances de calote por um fator de $e^{\beta_j}$ (a **Razão de Chances / Odds Ratio**).

In [1]:
# Importa bibliotecas numéricas e tabulares
import numpy as np
import pandas as pd

# Importa bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos de classificação e métricas do scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Tema visual padronizado
sns.set_theme(style="whitegrid", context="notebook")

## 4. Carregando os Dados: Dataset Default (ISLP)

O dataset **`Default`** contém dados de 10.000 clientes de cartão de crédito:
* **Alvo (`default`):** O cliente atrasou o pagamento (Yes/No)? (~3.3% são inadimplentes - classes desbalanceadas).
* **Preditores:**
  * `student`: É estudante (Yes/No)?
  * `balance`: Saldo médio devedor no cartão após a fatura.
  * `income`: Renda anual do cliente.

In [2]:
# Carregamento robusto: tenta ISLP, tenta GitHub ou gera dados sintéticos idênticos ao ISLP
try:
    from ISLP import load_data
    Default = load_data('Default')
except Exception:
    try:
        url = "https://raw.githubusercontent.com/intro-stat-learning/ISLP_labs/stable/ISLP/data/Default.csv"
        Default = pd.read_csv(url)
        if 'Unnamed: 0' in Default.columns:
            Default = Default.drop(columns=['Unnamed: 0'])
    except Exception:
        # Simulação com os parâmetros exatos do livro caso não haja conexão com a web
        np.random.seed(42)
        n = 10000
        student = np.random.choice(['No', 'Yes'], size=n, p=[0.7, 0.3])
        income = np.random.normal(33000, 13000, size=n)
        income = np.clip(income, 5000, 80000)
        balance = np.random.normal(800, 450, size=n) + (student == 'Yes') * 150
        balance = np.clip(balance, 0, 2700)
        
        # Equação logística baseada no modelo oficial do ISLP
        log_odds = -10.869 + 0.00573 * balance + 0.003 * (income / 1000) - 0.646 * (student == 'Yes')
        prob = 1 / (1 + np.exp(-log_odds))
        default = np.where(np.random.rand(n) < prob, 'Yes', 'No')
        Default = pd.DataFrame({'default': default, 'student': student, 'balance': balance, 'income': income})

# Codificação binária (0 e 1)
Default['default_num'] = (Default['default'] == 'Yes').astype(int)
Default['student_num'] = (Default['student'] == 'Yes').astype(int)

print(f"Tamanho do dataset: {Default.shape[0]} linhas e {Default.shape[1]} colunas")
print("Distribuição do alvo (Inadimplência):")
print(Default['default'].value_counts(normalize=True).map('{:.2%}'.format))
Default.head()

## 5. Como o modelo é treinado? (Máxima Verossimilhança)

Na Regressão Linear, minimizamos o erro de distância entre os pontos e a reta (RSS).
Na Regressão Logística, usamos **Máxima Verossimilhança (Maximum Likelihood Estimation - MLE)**.

### A Ideia Intuitiva:
O algoritmo procura os valores de $\beta_0, \beta_1$ que gerem probabilidades bem altas para quem realmente deu calote ($Y=1$) e probabilidades bem baixas para quem pagou em dia ($Y=0$):

$$\text{Verossimilhança } L(\boldsymbol{\beta}) = \prod_{\text{caloteiros}} p(x_i) \times \prod_{\text{bons pagadores}} (1 - p(x_i))$$

#### Exemplo Numérico Simples:
Suponha que temos 2 clientes no banco:
* **Cliente 1:** Deu calote real ($Y=1$).
* **Cliente 2:** Pagou em dia ($Y=0$).

Se o algoritmo testar uma curva com parâmetros que deem:
* $p(x_1) = 0{,}90$ (previu $90\%$ de calote para quem deu calote) $\rightarrow$ Acertou!
* $p(x_2) = 0{,}10$ (previu $10\%$ de calote para quem pagou em dia, logo $1 - 0{,}10 = 0{,}90$ de pagar) $\rightarrow$ Acertou!

A Verossimilhança é:
$$L = 0{,}90 \times (1 - 0{,}10) = 0{,}90 \times 0{,}90 = \mathbf{0{,}81}$$

Se ele testasse parâmetros ruins que dessem:
* $p(x_1) = 0{,}20$ e $p(x_2) = 0{,}80$, a conta seria:
$$L = 0{,}20 \times (1 - 0{,}80) = 0{,}20 \times 0{,}20 = \mathbf{0{,}04} \quad (valor baixo, modelo ruim)$$

> O computador utiliza otimizadores numéricos (como L-BFGS e Newton-Raphson) para achar automaticamente os betas que maximizam essa função.

In [3]:
# Preditores da Regressão Simples
X_simple = Default[['balance']]
y = Default['default_num']

# Divisão treino/teste (75% / 25%) estratificada
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y, test_size=0.25, random_state=42, stratify=y
)

# Treina o modelo logístico simples (C=1e5 remove penalidades de regularização para solução pura)
simple_logit = LogisticRegression(C=1e5, solver='lbfgs')
simple_logit.fit(X_train_s, y_train_s)

b0 = simple_logit.intercept_[0]
b1 = simple_logit.coef_[0][0]

print(f"Intercepto (beta_0): {b0:.4f}")
print(f"Coeficiente de saldo (beta_1): {b1:.6f}")
print(f"Multiplicador de chance a cada +$100 no saldo: {np.exp(b1 * 100):.3f}x")

In [4]:
# Grade de valores para plotar a curva predita suave
bal_grid = np.linspace(Default['balance'].min(), Default['balance'].max(), 300).reshape(-1, 1)
probs = simple_logit.predict_proba(bal_grid)[:, 1]

plt.figure(figsize=(10, 5))
plt.scatter(Default['balance'], Default['default_num'], alpha=0.15, color='steelblue', label='Clientes Reais (0 ou 1)')
plt.plot(bal_grid, probs, color='crimson', linewidth=3, label='Probabilidade Prevista P(Default=1)')
plt.axhline(0.5, color='black', linestyle='--', alpha=0.6, label='Limiar padrão de corte (0.50)')
plt.xlabel('Saldo Devedor do Cartão ($)')
plt.ylabel('Probabilidade Estimada de Inadimplência')
plt.title('Regressão Logística Simples: Ajuste da Sigmoide')
plt.legend()
plt.show()

## 6. Avaliação e o Impacto do Limiar de Decisão (*Threshold*)

O modelo logístico gera **probabilidades contínuas** (ex.: $0{,}12$, $0{,}48$, $0{,}76$).
Para tomar uma ação (ex.: bloquear cartão ou enviar para cobrança), precisamos definir uma linha de corte (limiar / *threshold*):

$$\text{Se } P(Y=1) \ge \text{Threshold} \rightarrow \text{Classifica como Caloteiro (1)}$$

### As 4 Possibilidades da Matriz de Confusão:
| | Previu: Não Inadimplente (0) | Previu: Inadimplente (1) |
| :--- | :---: | :---: |
| **Real: Não Inadimplente (0)** | **Verdadeiro Negativo (VN)** | **Falso Positivo (FP)** (Alarme Falso) |
| **Real: Inadimplente (1)** | **Falso Negativo (FN)** (Prejuízo!) | **Verdadeiro Positivo (VP)** (Detecção Correta) |

---

### Fórmulas Explicadas com Números:
Suponha uma matriz de teste com os seguintes resultados:
* $VN = 2400$, $FP = 15$
* $FN = 55$, $VP = 30$

1. **Acurácia (Total de Acertos / Total Geral):**
   $$\text{Acurácia} = \frac{VN + VP}{\text{Total}} = \frac{2400 + 30}{2500} = \frac{2430}{2500} = \mathbf{97{,}2\%}$$
   > *Atenção:* Parece ótimo, mas repare que dos 85 caloteiros reais ($55 + 30$), o banco deixou passar 55 caloteiros ilesos!

2. **Recall / Sensibilidade (Quantos caloteiros reais nós pegamos?):**
   $$\text{Recall} = \frac{VP}{VP + FN} = \frac{30}{30 + 55} = \frac{30}{85} = \mathbf{35{,}3\%}$$
   > O modelo deixou escapar quase $65\%$ dos inadimplentes!

3. **Precisão (Quando o modelo tocou o alarme, ele estava certo?):**
   $$\text{Precisão} = \frac{VP}{VP + FP} = \frac{30}{30 + 15} = \frac{30}{45} = \mathbf{66{,}7\%}$$

---

### A Decisão de Negócio: Mudando o Limiar de 0.50 para 0.20
Se baixarmos a régua de corte para $0{,}20$ ($20\%$ de probabilidade):
* Fica mais fácil classificar como risco $\rightarrow$ **Recall dispara (capturamos muito mais devedores)**.
* Em troca, aumentam os alarmes falsos (bloquear cartões de alguns clientes bons).

In [5]:
# Previsões de probabilidade no teste
y_prob_s = simple_logit.predict_proba(X_test_s)[:, 1]

# Testando dois limiares de decisão
y_pred_50 = (y_prob_s >= 0.50).astype(int)
y_pred_20 = (y_prob_s >= 0.20).astype(int)

cm_50 = confusion_matrix(y_test_s, y_pred_50)
cm_20 = confusion_matrix(y_test_s, y_pred_20)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(cm_50, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f"Limiar = 0.50 (Padrão)\nRecall: {recall_score(y_test_s, y_pred_50):.1%} | Inadimplentes Perdidos: {cm_50[1, 0]}")
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

sns.heatmap(cm_20, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title(f"Limiar = 0.20 (Conservador)\nRecall: {recall_score(y_test_s, y_pred_20):.1%} | Inadimplentes Perdidos: {cm_20[1, 0]}")
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

## 7. O que é a Curva ROC e a Métrica AUC?

Em vez de avaliar apenas um limiar fixo ($0{,}50$ ou $0{,}20$), a **Curva ROC** testa **todos os limiares possíveis** de $0\%$ a $100\%$ e desenha um gráfico:
* **Eixo Y (Benefício):** Taxa de Verdadeiro Positivo (Recall = quantos caloteiros pegamos).
* **Eixo X (Custo):** Taxa de Falso Positivo (quantos bons pagadores acusamos por engano).

### O que significa a área sob a curva (AUC - Area Under Curve)?
* $\text{AUC} = 0{,}50$: O modelo é equivalente a jogar uma moeda no ar (aleatório).
* $\text{AUC} = 1{,}00$: O modelo é perfeito e separa perfeitamente bons de maus pagadores em qualquer nível.
* $\text{AUC} \ge 0{,}90$: Excelente capacidade discriminativa!

## 8. Comparativo de Modelos: Regressão Logística vs. LDA vs. QDA vs. KNN

O livro ISLP compara quatro famílias clássicas de algoritmos:

1. **Regressão Logística Múltipla:** Modela diretamente a probabilidade via combinação linear sem assumir que os dados vêm de uma distribuição normal.
2. **LDA (Análise Discriminante Linear):** Modela a distribuição dos preditores assumindo que cada classe segue uma distribuição Normal (Gaussiana) com a **mesma matriz de covariância** (mesma variabilidade). Gera fronteiras lineares.
3. **QDA (Análise Discriminante Quadrática):** Mais flexível que o LDA. Assume que cada classe tem sua **própria matriz de covariância**, permitindo curvas e fronteiras parabólicas/curvas.
4. **KNN (K-Vizinhos Mais Próximos):** Abordagem **não paramétrica**. Não assume nenhuma fórmula matemática; apenas olha para os $K$ clientes mais próximos na base histórica e faz uma votação.

In [6]:
# Preditores multivariados
features = ['balance', 'income', 'student_num']
X_multi = Default[features]
y_multi = Default['default_num']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_multi, y_multi, test_size=0.25, random_state=42, stratify=y_multi
)

# Dicionário com os 4 competidores
models = {
    'Regressão Logística': LogisticRegression(C=1e5, solver='lbfgs'),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'KNN (K=10)': KNeighborsClassifier(n_neighbors=10)
}

metrics_summary = {}

plt.figure(figsize=(9, 6))

for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    y_prob = clf.predict_proba(X_te)[:, 1]
    y_pred = clf.predict(X_te)
    
    auc = roc_auc_score(y_te, y_prob)
    rec = recall_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    acc = accuracy_score(y_te, y_pred)
    
    metrics_summary[name] = {
        'ROC-AUC': f"{auc:.4f}",
        'Recall (corte 0.5)': f"{rec:.2%}",
        'Precisão (corte 0.5)': f"{prec:.2%}",
        'Acurácia': f"{acc:.2%}"
    }
    
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.7, label='Referência Aleatória (AUC = 0.500)')
plt.xlabel('Taxa de Falso Alarme (Falso Positivo)')
plt.ylabel('Taxa de Captura de Inadimplência (Recall)')
plt.title('Comparativo de Performance: Curvas ROC')
plt.legend()
plt.show()

# Exibe a tabela consolidada de desempenho
pd.DataFrame(metrics_summary).T

## 9. O Paradoxo do Estudante (Paradoxo de Simpson)

Quando olhamos os coeficientes da Regressão Logística Múltipla, encontramos um fenômeno surpreendente:

In [7]:
logit_m = models['Regressão Logística']

coef_df = pd.DataFrame({
    'Variável': features,
    'Coeficiente (Beta)': logit_m.coef_[0],
    'Multiplicador de Chance exp(Beta)': np.exp(logit_m.coef_[0])
}).sort_values(by='Coeficiente (Beta)', ascending=False)

print(f"Intercepto base: {logit_m.intercept_[0]:.4f}")
coef_df

### Por que o coeficiente de `student` ficou negativo no modelo múltiplo?

1. **Análise Isolada (Univariada):**
   Se rodarmos uma regressão olhando apenas para `student`, o coeficiente é **positivo** (estudantes no geral atrasam mais faturas do que não estudantes).
2. **Análise Conjunta (Multivariada):**
   Quando incluímos o saldo (`balance`) no modelo, o coeficiente de `student` vira **negativo** ($e^{\beta} < 1$).

**A Explicação Intuitiva:**
* Estudantes geralmente têm faturas mais altas de cartão (carregam mais saldo).
* Porém, se compararmos **um estudante e um não estudante com o mesmíssimo saldo devedor** (ex.: ambos devendo exatamente $\$1.500$), o estudante tem **menor** probabilidade de dar calote do que o não estudante!
* Isso ocorre porque a renda média do estudante é menor, mas ele é subsidiado pela família ou tem menor taxa de inadimplência dentro daquela mesma faixa de dívida.

> Esse é um exemplo clássico do **Paradoxo de Simpson**: olhar variáveis isoladas pode induzir a conclusões exatamente opostas à realidade!

## 10. Resumo Executivo para Tomada de Decisão (Visão de Negócio)

* **Objetivo:** Automatizar a avaliação de risco de crédito e detectar clientes com alta probabilidade de inadimplência.
* **Driver de Risco Principal:** O saldo devedor mensal remanescente (`balance`) é a variável que domina o risco. Renda pessoal tem impacto secundário marginal.
* **Comparação Técnica:** Os modelos lineares paramétricos (**Regressão Logística** e **LDA**) atingiram o melhor desempenho (ROC-AUC $\approx 0{,}95$), superando o KNN (que sofre com a dispersão dos dados nas pontas).
* **Recomendação Estratégica de Calibração:** Não utilizar a linha de corte padrão de $0{,}50$ em produção, pois ela deixa passar muitos inadimplentes. Calibrar o sistema com um limiar entre $0{,}20$ e $0{,}25$, garantindo uma captura preventiva de risco superior a $70\%$ com atrito mínimo a bons clientes.